# Exercício 08 — GovBot Cidadão

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## 1. Base documental fictícia

In [ ]:
DOCS = {
    "iptu": "IPTU 2026: 1ª parcela até 10/janeiro; desconto de 5% pagamento único.",
    "iluminacao": "Reportar poste: linha verde 0800-XXX com código do poste.",
    "alvara": "Alvará de funcionamento: protocolo digital com planta e RRMC.",
}


## 2. Classificação + RAG simplificado

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
import os

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.2,
)

rotulo_p = ChatPromptTemplate.from_messages([
    ("system", "Classifica em uma etiqueta: imposto | protocolo | servico_urbano | licenca"),
    ("human", "Pergunta: {q}"),
])
rotulo = (rotulo_p | llm | StrOutputParser()).invoke({"q": "Quando vence o IPTU?"})
print("Classe:", rotulo)

ctx = "\n".join(f"{k}: {v}" for k, v in DOCS.items())
resp_p = ChatPromptTemplate.from_messages([
    ("system", "Responde com base nos documentos; cita a secção lógica. Português europeu."),
    ("human", "Docs:\n{ctx}\n\nPergunta: {q}"),
])
print((resp_p | llm | StrOutputParser()).invoke({"ctx": ctx, "q": "Como peço reembolso de luz?"}))


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
